In [1]:
!nvidia-smi

Tue Apr 28 13:43:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.86                 Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   40C    P8              2W /   35W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate


Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)

   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [tran

In [4]:
pip install evaluate -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, set_seed, AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
from datasets import load_dataset, load_from_disk
import evaluate # Import the evaluate library
import matplotlib.pyplot as plt
import pandas as pd

nltk.download("punkt")

e:\anaconda\anaconda_nav\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\belal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

e:\anaconda\anaconda_nav\envs\textS\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\belal\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\anaconda\anaconda_nav\envs\textS\lib\site-packages\huggingface_hub\file_download.py:805: UserWarning:

KeyboardInterrupt: 

In [ ]:
#download & unzip data

!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip

--2026-04-27 10:30:00--  https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip [following]
--2026-04-27 10:30:01--  https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip’

summarizer-data.zip 100%[===================>]   7.54M  5.57MB/s    in 1.4s    

2026-04-27 10:30:02 (5.57 MB/s) - ‘summarizer-data.zip’ saved [7903594/790

In [ ]:
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [ ]:
split_lengths =[len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue: ")

print(dataset_samsum["test"] [1]["dialogue"])

print("/nSummary: ")

print(dataset_samsum["test"] [1]["summary"])


Split lengths: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']

Dialogue: 
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)
/nSummary: 
Eric and Rob are going to watch a stand-up on youtube.


In [ ]:
def convert_examples_to_features(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )

  target_encodings = tokenizer(example_batch['summary'], max_length = 128, truncation=True)

  encodings = {
      'input_ids': input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels': target_encodings['input_ids']
  }

  return encodings

In [ ]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [ ]:
dataset_samsum_pt['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [ ]:
# Training

from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# 3. Define the Training Arguments
trainer_args = Seq2SeqTrainingArguments(
    # --- File System & Pathing ---
    output_dir='pegasus-samsum',          # Where model checkpoints and logs are saved.

    # --- Optimization & Schedule ---
    num_train_epochs=1,                   # Total number of times the model sees the whole dataset.
    learning_rate=5e-5,                   # The starting "step size" for weight updates.
    warmup_steps=500,                     # Linearly increase learning rate from 0 to 5e-5 over 500 steps.
    weight_decay=0.01,                    # L2 regularization to prevent weights from becoming too large.

    # --- Hardware & Memory Management ---
    per_device_train_batch_size=1,        # Samples processed per GPU per step. 1 is safe for Pegasus.
    per_device_eval_batch_size=1,         # Samples processed during evaluation.
    gradient_accumulation_steps=16,       # Wait for 16 steps before updating weights (Effective Batch = 16).
    fp16=True,                            # Use Mixed Precision (16-bit) to save memory and speed up training.

    # --- Evaluation & Logging ---
    eval_strategy='steps',          # Evaluate based on steps rather than epochs.
    eval_steps=500,                       # Run evaluation every 500 steps.
    logging_steps=10,                     # Print training loss to the console every 10 steps.

    # --- Seq2Seq Specifics ---
    predict_with_generate=True,           # Required for summarization! It tells the model to
                                          # actually "write" summaries during evaluation.
    generation_max_length=128,            # Limit the length of summaries during evaluation.

    # --- Saving Strategy ---
    save_steps=1e6,                       # Essentially disables mid-training saving to save disk space.
    save_total_limit=2,                   # Only keep the 2 most recent checkpoints.
)

# 4. Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model_pegasus,
    args=trainer_args,
    # In latest 2026 versions, use 'processing_class' if 'tokenizer' throws a TypeError
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"]
)

# 5. Launch Training
# trainer.train()

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
500,27.152789,1.495244
921,25.658951,1.440938


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=921, training_loss=29.812008270611592, metrics={'train_runtime': 2702.2536, 'train_samples_per_second': 5.452, 'train_steps_per_second': 0.341, 'total_flos': 5535530955915264.0, 'train_loss': 29.812008270611592, 'epoch': 1.0})

In [ ]:
from tqdm import tqdm

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches that we can process simultaneously.
    Yields successive batch-sized chunks from list_of_elements.
    """
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]


def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=16,
    device=device,
    column_text="article",
    column_summary="highlights"
):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches),
        total=len(article_batches)
    ):
        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        # Decode generated summaries
        decoded_summaries = [
            tokenizer.decode(
                summary,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )
            for summary in summaries
        ]

        # Add predictions and references to the metric
        metric.add_batch(
            predictions=decoded_summaries,
            references=target_batch
        )

    # Compute final score
    score = metric.compute()
    return score


    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)


        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
  
        rouge_metric = load_metric('rouge')

        score = self.calculate_metric_on_test_ds(
        dataset_samsum_pt['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
            )

        rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )

        df = pd.DataFrame(rouge_dict, index = ['pegasus'] )
        df.to_csv(self.config.metric_file_name, index=False)



In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_samsum["test"][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text="dialogue", column_summary="summary"
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

pd.DataFrame(rouge_dict, index = [f'pegasus'])

100%|██████████| 5/5 [00:17<00:00,  3.40s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.417832,0.171995,0.293534,0.293277


### Load Model and Tokenizer for Inference

Now, let's use the loaded model and tokenizer to generate a summary for a new dialogue.

In [ ]:
# Saving the model

model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Saving the tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

### Download Model and Tokenizer to Local Machine

To download the `pegasus-samsum-model` and `tokenizer` directories, we'll first compress them into `.zip` files and then use `google.colab.files.download`.

In [ ]:
# Zip the model directory
!zip -r pegasus-samsum-model.zip pegasus-samsum-model

# Zip the tokenizer directory
!zip -r tokenizer.zip tokenizer

  adding: pegasus-samsum-model/ (stored 0%)
  adding: pegasus-samsum-model/model.safetensors (deflated 7%)
  adding: pegasus-samsum-model/config.json (deflated 61%)
  adding: pegasus-samsum-model/generation_config.json (deflated 40%)
  adding: tokenizer/ (stored 0%)
  adding: tokenizer/tokenizer.json (deflated 78%)
  adding: tokenizer/tokenizer_config.json (deflated 78%)


In [ ]:
from google.colab import files

# Download the model zip file
files.download('pegasus-samsum-model.zip')

# Download the tokenizer zip file
files.download('tokenizer.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}




sample_text = dataset_samsum["test"][0]["dialogue"]
reference = datasest_samsum["test"][0]["summary"]


pipe = pipeline("summarization", model= "pegasus-samsum-model", tokenizer=tokenizer)

##
print("Dialogue:")
print(sample_text)

##
print("\nReference Summary:")
print(reference)

##
print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])